# Investigation du signal résiduel identifié par SHAP

SHAP a révélé que 'step' (temps) est la feature la plus influente, alors
que sa corrélation linéaire était quasi nulle -> signal non-linéaire à
caractériser. Ce script :
  1. Recalcule l'importance native XGBoost en mode 'gain' (comparaison
     plus juste avec SHAP que le mode 'weight' par défaut)
  2. Isole et visualise la relation non-linéaire entre step et isFraud

Auteur : Rasmané

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

DOSSIER = r"C:\Users\hp\Documents\Fraude_detection\data"
DOSSIER_SORTIE = r"C:\Users\hp\Documents\Fraude_detection\data"

modele = joblib.load(f"{DOSSIER}/modele_xgboost_optimise.joblib")
X_train = pd.read_csv(f"{DOSSIER}/X_train.csv")
y_train = pd.read_csv(f"{DOSSIER}/y_train.csv").squeeze()

## 1. IMPORTANCE NATIVE EN MODE 'GAIN' (comparaison juste avec SHAP)

In [ ]:
print("=" * 70)
print("1. IMPORTANCE NATIVE XGBOOST - MODE 'GAIN'")
print("=" * 70)

booster = modele.get_booster()
scores_gain = booster.get_score(importance_type="gain")

importance_gain = pd.Series(scores_gain).reindex(X_train.columns).fillna(0)
importance_gain = importance_gain.sort_values(ascending=False)
print(importance_gain)

print("\n-> Compare ce classement à celui de SHAP (step, amount, oldbalanceOrg")
print("   en tête). Une meilleure concordance ici confirmerait que le mode")
print("   'weight' par défaut donnait une image trompeuse.")

## 2. RELATION step / isFraud - NON LINÉAIRE

In [ ]:
print("\n" + "=" * 70)
print("2. TAUX DE FRAUDE PAR STEP (sur train)")
print("=" * 70)

taux_par_step = pd.concat([X_train["step"], y_train], axis=1).groupby("step")["isFraud"].agg(
    ["mean", "count"]
)
taux_par_step.columns = ["taux_fraude", "nb_transactions"]

print(f"Step avec le taux de fraude le plus élevé :")
print(taux_par_step.sort_values("taux_fraude", ascending=False).head(10))

print(f"\nStep avec le taux de fraude le plus bas (hors steps à 0 transaction) :")
print(taux_par_step[taux_par_step["nb_transactions"] > 50].sort_values("taux_fraude").head(10))

# Écart-type du taux de fraude entre steps -> mesure la variabilité temporelle
ecart_type_taux = taux_par_step["taux_fraude"].std()
taux_moyen = y_train.mean()
print(f"\nTaux de fraude moyen global : {taux_moyen*100:.4f}%")
print(f"Écart-type du taux de fraude entre steps : {ecart_type_taux*100:.4f} points")
print(f"Coefficient de variation : {ecart_type_taux/taux_moyen:.2f}")

# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
taux_par_step["taux_fraude"].plot(ax=axes[0], color="#E63946")
axes[0].axhline(y=taux_moyen, color="#2E86AB", linestyle="--", label="Taux moyen global")
axes[0].set_title("Taux de fraude par step")
axes[0].set_xlabel("step")
axes[0].legend()

# Découpage en deciles de step pour lisser et voir la tendance
X_train_copy = X_train.copy()
X_train_copy["isFraud"] = y_train
X_train_copy["decile_step"] = pd.qcut(X_train_copy["step"], 10, duplicates="drop")
taux_par_decile = X_train_copy.groupby("decile_step", observed=True)["isFraud"].mean()
taux_par_decile.plot(kind="bar", ax=axes[1], color="#F4A261")
axes[1].axhline(y=taux_moyen, color="#2E86AB", linestyle="--")
axes[1].set_title("Taux de fraude par décile de step")
axes[1].set_xlabel("Décile de step")
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/investigation_step_fraude.png", dpi=120)
#plt.close()
print("\nGraphique sauvegardé : investigation_step_fraude.png")

## 3. RELATION amount / oldbalanceOrg - MAGNITUDE POUR FRAUDE VS NON-FRAUDE

In [ ]:
print("\n" + "=" * 70)
print("3. DISTRIBUTIONS amount / oldbalanceOrg PAR CLASSE (rappel, log)")
print("=" * 70)
for col in ["amount", "oldbalanceOrg", "montant_moyen_dest"]:
    stats = pd.concat([X_train[col], y_train], axis=1).groupby("isFraud")[col].describe()
    print(f"\n--- {col} ---")
    print(stats.round(2))

print("""
NOTE : si les distributions de amount/oldbalanceOrg/montant_moyen_dest
diffèrent sensiblement entre fraude et non-fraude (médiane, quartiles),
même sans corrélation linéaire ni seuil net, cela explique pourquoi un
modèle à forte capacité (profondeur 10) peut exploiter ce signal diffus
alors qu'un arbre simple ou une corrélation de Pearson ne le détectent
pas.
""")